
## Transform circuits data
1. Read bronze _`sprints`_ table
2. Keep only columns that are required (remove url col)
3. Standardize column names using snake_case(constructorId -> constructor_id, driverId -> driver_id,raceName -> race_name, positionText -> finsih_position_text)
4. rename to give meaningful names(date -> race_date, grid -> grid_position, laps -> completed_laps, number -> car_number, position -> finish_position) 
6. filter out rows where season, round construcor_id or  driver_id is null
6. Remove duplicate rows
7. Transform values of _race_name_ to Title Case 
8. Write the tranformed data to silver table 

In [0]:
%run ../00-common/01.environment-config

In [0]:
%python
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"


In [0]:
from pyspark.sql import functions as F

**Steps 1 to 4  Read and change schema**

In [0]:
# circuits_df = spark.read.option('VersionAsOf',0).table(bronze_table)
# use spark.read.table if you need additional options, otherwise use spark.table for simplicity

In [0]:
sprints_df = (spark
              .table(bronze_table)
              .drop(F.col('url'))
              .withColumnsRenamed(
                {"date":"results_date",
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "grid":"grid_positions",
                "laps":"completed_laps",
                "number": "car_number",
                "position":"position_finished",
                "positionText":"finsih_position_text"
                })
)


**Step 5 and 6: Apply data qulaity checks**
- remove nulls
- remove duplicates

In [0]:
sprints_valid_df = (
    sprints_df
    .filter(
        F.col("season").isNotNull()& 
        F.col("round").isNotNull()&
        F.col("constructor_id").isNotNull()&
        F.col("driver_id").isNotNull())
    .dropDuplicates(["season","round","constructor_id","driver_id"])
    
    )
    

In [0]:
display(sprints_df.count()-sprints_valid_df.count())

**7. Transform values of _race_name_ to Title Case**


In [0]:
sprints_final_df = (
    sprints_valid_df
    .withColumn("race_name", F.initcap(F.col("race_name")))

)

In [0]:
display(sprints_final_df)

**8. Write the tranformed data to silver table**

In [0]:
(sprints_final_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)
 
 

In [0]:
display(spark.table(silver_table))